In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Forces TensorFlow to pretend no GPU exists

import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras import mixed_precision

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

In [ ]:
import os
import random
from pathlib import Path
import cv2

DATASET_ROOT = Path("../datasets/FaceForensics++_C23/")
DEST_DIR = Path("../datasets/FF_extract_1v0")

VAL_THRESHOLD = 800
TEST_THRESHOLD = 950

FRAME_SKIP = 30
MAX_VIDEOS_PER_FOLDER = 1000

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def prepare_directory_structure(base_dir: Path, subsets: list, categories: list):
    for subset in subsets:
        for category in categories:
            (base_dir / subset / category).mkdir(parents=True, exist_ok=True)


def get_dataset_sources(root_path: Path, exclude):
    source_real = None
    sources_fake = []

    for dirpath, _, _ in os.walk(root_path):
        # print(f"jsdhkjdhdj {dirpath}")
        current_dir = Path(dirpath)
        dir_name = current_dir.name

        if dir_name == "original" and source_real is None:
            # print('sjhddghj')
            source_real = current_dir
        elif dir_name not in [exclude, "csv", "FaceForensics++_C23" ]:
            if current_dir != root_path:
                sources_fake.append(current_dir)

    return source_real, sources_fake


def get_subset_by_id(video_id: int, val_limit: int, test_limit: int) -> str:
    if video_id < val_limit:
        return "training"
    elif video_id < test_limit:
        return "validation"
    return "test"

import urllib

YUNET_MODEL_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
YUNET_MODEL_PATH = "face_detection_yunet_2023mar.onnx"


def get_yunet_detector():
    if not os.path.exists(YUNET_MODEL_PATH):
        print(f"Downloading YuNet ({YUNET_MODEL_PATH})...")
        urllib.request.urlretrieve(YUNET_MODEL_URL, YUNET_MODEL_PATH)
        print("Done")

    detector = cv2.FaceDetectorYN.create(
        model=YUNET_MODEL_PATH,
        config="",
        input_size=(320, 320),
        score_threshold=0.6, 
        nms_threshold=0.3,
    )
    return detector


def extract_faces_from_folder(
    video_folder: Path | str,
    dest_dir: Path | str,
    category_name: str,
    max_videos: int = 1000,
    frames_per_video: int = 3, 
    val_limit: int = 800,
    test_limit: int = 950,
    target_size: tuple[int, int] = (299, 299), 
    margin_ratio: float = 0.3,
):
    video_folder = Path(video_folder)
    dest_dir = Path(dest_dir)

    videos = [f for f in video_folder.iterdir() if f.suffix.lower() == ".mp4"]

    # selected_videos = random.sample(videos, min(max_videos, len(videos)))
    saved_count = 0

    detector = get_yunet_detector()
    video_cnt=0

    for vid_path in videos:
        try:
            if "DeepFakeDetection" in str(vid_path):
                video_id=video_cnt
            else:
                video_id = int(vid_path.stem.split("_")[0])
        except ValueError:
            print(f"{vid_path.name} empty, skip.")
            continue

        video_cnt += 1
        cap = cv2.VideoCapture(str(vid_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames < 10:
            cap.release()
            continue

        frame_indices = [
            int(total_frames * (i + 1) / (frames_per_video + 1))
            for i in range(frames_per_video)
        ]

        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()
            if not success or frame is None:
                continue

            h_img, w_img, _ = frame.shape
            detector.setInputSize((w_img, h_img))
            _, faces = detector.detect(frame)

            if faces is not None and len(faces) > 0:
                x, y, w, h = map(int, faces[0][:4])

                margin_x = int(w * margin_ratio)
                margin_y = int(h * margin_ratio)
                x1 = max(0, x - margin_x)
                y1 = max(0, y - margin_y)
                x2 = min(w_img, x + w + margin_x)
                y2 = min(h_img, y + h + margin_y)

                face_img = frame[y1:y2, x1:x2]

                if face_img.size == 0:
                    continue

                face_img = cv2.resize(
                    face_img, target_size, interpolation=cv2.INTER_CUBIC
                )

                subset_name = get_subset_by_id(video_id, val_limit, test_limit)
                output_filename = (
                    f"{video_folder.name}_{vid_path.stem}_f{frame_idx}.jpg"
                )
                save_path = (
                    dest_dir / subset_name / category_name / output_filename
                )

                cv2.imwrite(str(save_path), face_img)
                saved_count += 1

        cap.release()

    print(f"Extracted {saved_count} faces with {margin_ratio*100:.0f}% margin to '{dest_dir}'.")

In [ ]:
source_real, sources_fake = get_dataset_sources(DATASET_ROOT, exclude="All")

if not source_real or not sources_fake:
    print("Folders not found")
else:
    print(f"Real videos: {source_real}")
    print(f"Fake sources: {len(sources_fake)}, methods: {sources_fake}")

    subsets = ["training", "validation", "test"]
    categories = ["0Real", "1Fake"]
    prepare_directory_structure(DEST_DIR, subsets, categories)

    print("\nStart extraction...")

    extract_faces_from_folder(
        video_folder=source_real,
        dest_dir=DEST_DIR,
        category_name="0Real",
        max_videos=MAX_VIDEOS_PER_FOLDER,
        # frame_skip=FRAME_SKIP,
        val_limit=VAL_THRESHOLD,
        test_limit=TEST_THRESHOLD,
    )

    for fake_folder in sources_fake:
        extract_faces_from_folder(
            video_folder=fake_folder,
            dest_dir=DEST_DIR,
            category_name="1Fake",
            max_videos=MAX_VIDEOS_PER_FOLDER,
            # frame_skip=FRAME_SKIP,
            val_limit=VAL_THRESHOLD,
            test_limit=TEST_THRESHOLD,
        )

    print("\nDone")

In [ ]:
# print(source_real)

../datasets/FaceForensics++_C23/original


In [ ]:
from pathlib import Path

def crop_faces_to_eval2(
    input_folder: Path | str,
    output_dir: Path | str = "./datasets/FF_extract_1v0/eval",
    category_name: str = "1Fake", 
    frames_per_video: int = 3,
    test_limit: int = 950,
    target_size: tuple[int, int] = (299, 299),
    margin_ratio: float = 0.3,
):
    input_folder = Path(input_folder)
    dest_dir = Path(output_dir)
    print("Making")
    dest_dir.mkdir(parents=True, exist_ok=True)


    videos = [f for f in input_folder.iterdir() if f.suffix.lower() == ".mp4"]
    video_cnt=0
    detector = get_yunet_detector()
    saved_count = 0

    for vid_path in videos:
        try:
            if "DeepFakeDetection" in str(vid_path):
                video_id=video_cnt
            else:
                video_id = int(vid_path.stem.split("_")[0])
        except ValueError:
            print(f"Not found: {vid_path.name}, skip.")
            continue

        video_cnt += 1
        if video_id < test_limit:
            continue

        cap = cv2.VideoCapture(str(vid_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < 10:
            cap.release()
            continue

        frame_indices = [
            int(total_frames * (i + 1) / (frames_per_video + 1))
            for i in range(frames_per_video)
        ]

        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()
            if not success or frame is None:
                continue

            h_img, w_img, _ = frame.shape
            detector.setInputSize((w_img, h_img))
            _, faces = detector.detect(frame)

            if faces is not None and len(faces) > 0:
                x, y, w, h = map(int, faces[0][:4])

                margin_x = int(w * margin_ratio)
                margin_y = int(h * margin_ratio)
                x1 = max(0, x - margin_x)
                y1 = max(0, y - margin_y)
                x2 = min(w_img, x + w + margin_x)
                y2 = min(h_img, y + h + margin_y)

                face_img = frame[y1:y2, x1:x2]

                if face_img.size == 0:
                    continue

                face_img = cv2.resize(
                    face_img, target_size, interpolation=cv2.INTER_CUBIC
                )

                output_filename = (
                    f"{input_folder.name}_{vid_path.stem}_f{frame_idx}.jpg"
                )
                save_path = (
                    dest_dir / category_name / output_filename
                )

                save_path.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(save_path), face_img)
                saved_count += 1

        cap.release()

    print(
        f"Extracted {saved_count} faces with( {margin_ratio*100:.0f}% margin to '{dest_dir}'."
    )

In [ ]:
crop_faces_to_eval2(
    input_folder="./datasets/FaceForensics++_C23/NeuralTextures",
    output_dir="./datasets/FF_extract_1v0/eval_balanced",
    category_name="1Fake",
    # img_size=(150, 150),
)

In [ ]:
from pathlib import Path
import cv2

def crop_faces_to_eval(
    input_folder: str | Path,
    output_dir: str | Path = "./datasets/FF_extract_1v0/eval",
    category_name: str = "0Real", 
    img_size: tuple[int, int] = (299, 299), 
    margin_ratio: float = 0.3,  
):
    """
    Worse than v2, takes video randomly, does not distinguish betweed vid_id + only 1 frame per video
    """
    input_path = Path(input_folder)
    save_path = Path(output_dir) / category_name
    save_path.mkdir(parents=True, exist_ok=True)

    files = [f for f in input_path.iterdir()]

    if not files:
        print(f"'{input_path}' empty")
        return

    detector = get_yunet_detector()
    saved_count = 0

    for file_path in files:
        frame = None
        if file_path.suffix.lower() in {".mp4", ".avi", ".mov"}:
            cap = cv2.VideoCapture(str(file_path))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            if total_frames > 0:
                cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)

            success, frame = cap.read()
            cap.release()

            if not success or frame is None:
                continue
        else:
            frame = cv2.imread(str(file_path))

        if frame is None:
            continue

        h_img, w_img, _ = frame.shape
        detector.setInputSize((w_img, h_img))
        _, faces = detector.detect(frame)

        if faces is not None and len(faces) > 0:
            x, y, w, h = map(int, faces[0][:4])

            margin_x = int(w * margin_ratio)
            margin_y = int(h * margin_ratio)
            x1 = max(0, x - margin_x)
            y1 = max(0, y - margin_y)
            x2 = min(w_img, x + w + margin_x)
            y2 = min(h_img, y + h + margin_y)

            face_img = frame[y1:y2, x1:x2]

            if face_img.size == 0:
                continue
            face_img = cv2.resize(
                face_img, img_size, interpolation=cv2.INTER_CUBIC
            )

            out_filename = f"{input_path.name}_{file_path.stem}.jpg"
            output_file_path = save_path / out_filename

            cv2.imwrite(str(output_file_path), face_img)
            saved_count += 1

    print(
        f"Extracted {saved_count} faces to '{save_path}' (size: {img_size}, margin: {margin_ratio*100:.0f}%)"
    )

crop_faces_to_eval(
    input_folder="./datasets/FaceForensics++_C23/NeuralTextures",
    output_dir="./datasets/FF_extract_1v0/eval",
    category_name="1Fake",
    # img_size=(150, 150),
)